In [ ]:

import json, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

ID="id"; TARGET="health_condition"; SEED=2027; FOLDS=5
C=np.array(["at-risk","fit","unhealthy"]); M={c:i for i,c in enumerate(C)}
base=Path("/kaggle/input/competitions/playground-series-s6e7")
tr=pd.read_csv(base/"train.csv"); te=pd.read_csv(base/"test.csv"); y=tr[TARGET].map(M).to_numpy()

def states(d):
    z=pd.DataFrame(index=d.index)
    s=d.sleep_duration
    z["sleep_zone"]=pd.cut(s,[-np.inf,6,7,np.inf],right=False,labels=["lt6","6to7","ge7"]).astype("object").fillna("NA")
    for c in ["stress_level","physical_activity_level","sleep_quality","diet_type","gender","smoking_alcohol"]:
        z[c]=d[c].fillna("NA").astype(str)
    z["pattern"]=d[["sleep_duration","stress_level","physical_activity_level"]].isna().astype(int).astype(str).agg("".join,axis=1)
    z["core"]=z.sleep_zone+"|"+z.stress_level+"|"+z.physical_activity_level
    z["core_sleepq"]=z.core+"|"+z.sleep_quality
    z["fallback"]=z.sleep_zone+"|"+z.stress_level
    return z

A=states(tr); B=states(te)
keys=[
 ["core_sleepq"],["core"],["sleep_zone","stress_level"],
 ["sleep_zone","physical_activity_level","pattern"],
 ["stress_level","physical_activity_level","pattern"],["pattern"],
]

def lookup_fit_predict(a_train,y_train,a_query,alpha=8.0):
    prior=np.bincount(y_train,minlength=3).astype(float); prior/=prior.sum()
    out=np.tile(prior,(len(a_query),1)); unresolved=np.ones(len(a_query),bool)
    for cols in keys:
        tmp=a_train[cols].copy()
        tmp["_y"]=y_train
        counts=tmp.groupby(cols,dropna=False)["_y"].value_counts().unstack(fill_value=0)
        for k in range(3):
            if k not in counts: counts[k]=0
        counts=counts[[0,1,2]]
        probs=(counts+alpha)/(counts.sum(1).to_numpy()[:,None]+3*alpha)
        q=a_query[cols].merge(probs,left_on=cols,right_index=True,how="left")[[0,1,2]].to_numpy(float)
        ok=unresolved & np.isfinite(q).all(1)
        out[ok]=q[ok]; unresolved[ok]=False
    return out

oof=np.zeros((len(tr),3)); pred=np.zeros((len(te),3)); fs=[]; t0=time.time()
skf=StratifiedKFold(FOLDS,shuffle=True,random_state=SEED)
for f,(i,j) in enumerate(skf.split(A,y),1):
    oof[j]=lookup_fit_predict(A.iloc[i],y[i],A.iloc[j])
    pred+=lookup_fit_predict(A.iloc[i],y[i],B)/FOLDS
    s=balanced_accuracy_score(y[j],oof[j].argmax(1));fs.append(float(s));print(f,s)
overall=float(balanced_accuracy_score(y,oof.argmax(1)))
od=pd.DataFrame({ID:tr[ID]});td=pd.DataFrame({ID:te[ID]})
for k,c in enumerate(C):od[c]=oof[:,k];td[c]=pred[:,k]
od.to_csv("oof_preds.csv",index=False);td.to_csv("test_preds.csv",index=False)
pd.DataFrame({ID:te[ID],TARGET:C[pred.argmax(1)]}).to_csv("submission.csv",index=False)
Path("training_summary.json").write_text(json.dumps({"experiment":"conditional_rule_lookup","fold_scores":fs,"oof_balanced_accuracy":overall,"elapsed_minutes":(time.time()-t0)/60},indent=2))
print(overall)
